# 73_all_pairs_shortest_paths

Audience: junior researcher

- Challenge path: `challenges/hard/73_all_pairs_shortest_paths`
- Source spec: [challenges/hard/73_all_pairs_shortest_paths/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/73_all_pairs_shortest_paths/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Given a weighted directed graph of <code>N</code> vertices represented as an
  <code>N</code> &times; <code>N</code> distance matrix, compute the shortest path distance between
  every pair of vertices using the Floyd-Warshall algorithm. The matrix is stored as a flat array in
  row-major order: <code>dist[i * N + j]</code> is the weight of the directed edge from vertex
  <code>i</code> to vertex <code>j</code>. A value of <code>+infinity</code> means no direct edge
  exists. The diagonal is always zero. For each intermediate vertex <code>k</code> from <code>0</code> to <code>N - 1</code>
  (in order), update all pairs:
</p>
<p>
  \[
    \text{output}[i][j] = \min\!\bigl(\text{output}[i][j],\;
      \text{output}[i][k] + \text{output}[k][j]\bigr)
    \quad \forall\, i, j
  \]
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
</ul>

<h2>Example:</h2>
<pre>
Input: N = 4
dist = [
  0,   5, inf,  10,   // row 0: edges from vertex 0
  inf, 0,   3, inf,   // row 1: edges from vertex 1
  inf, inf, 0,   1,   // row 2: edges from vertex 2
  inf, inf, inf, 0    // row 3: edges from vertex 3
]

Output:
output = [
  0,   5,   8,   9,   // shortest paths from vertex 0
  inf, 0,   3,   4,   // shortest paths from vertex 1
  inf, inf, 0,   1,   // shortest paths from vertex 2
  inf, inf, inf, 0    // shortest paths from vertex 3
]

Explanation:
- output[0][2] = 8   (path 0 -&gt; 1 -&gt; 2, cost 5 + 3 = 8)
- output[0][3] = 9   (path 0 -&gt; 1 -&gt; 2 -&gt; 3, cost 5 + 3 + 1 = 9, beats direct 0 -&gt; 3 = 10)
- output[1][3] = 4   (path 1 -&gt; 2 -&gt; 3, cost 3 + 1 = 4)
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 4,096</li>
  <li>Edge weights are finite <code>float32</code> values or <code>+infinity</code> (no edge)</li>
  <li>The input contains no negative cycles</li>
  <li>The diagonal satisfies <code>dist[i * N + i] = 0</code> for all <code>i</code></li>
  <li><code>dist</code> and <code>output</code> are flat arrays of <code>N &times; N</code> floats in row-major order</li>
  <li>Performance is measured with <code>N</code> = 2,048</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/73_all_pairs_shortest_paths/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    d = dist.view(N, N).clone()
    for k in range(N):
        d = torch.minimum(d, d[:, k : k + 1] + d[k : k + 1, :])
    output.copy_(d.view(-1))


## Jax

Source: `challenges/hard/73_all_pairs_shortest_paths/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

try:
    import jax
    import jax.numpy as jnp
    from jax import lax
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None
    lax = None

import torch


def _solve_impl(dist, N: int):
    if jax is None:
        d = dist.view(N, N).clone()
        for k in range(N):
            d = torch.minimum(d, d[:, k : k + 1] + d[k : k + 1, :])
        return d.view(-1)

    d = jnp.reshape(dist, (N, N))

    def body(k, cur):
        return jnp.minimum(cur, cur[:, k : k + 1] + cur[k : k + 1, :])

    d = lax.fori_loop(0, N, body, d)
    return jnp.reshape(d, (-1,))


if jax is None:

    def solve(dist, N: int):
        return _solve_impl(dist, N)

else:
    solve = jax.jit(_solve_impl, static_argnames=("N",))


## Triton

Source: `challenges/hard/73_all_pairs_shortest_paths/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    d = dist.view(N, N).clone()
    for k in range(N):
        d = torch.minimum(d, d[:, k : k + 1] + d[k : k + 1, :])
    output.copy_(d.view(-1))


## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.